# Notebook 06 — SQL Queries

**Project:** Programme Health Intelligence: A Learning Analytics Case Study  
**Dataset:** Open University Learning Analytics Dataset (OULAD)  
**Author:** Megha Sinha  
**Date:** June 2026

---

## Objective

This notebook demonstrates operational SQL queries against the OULAD
dataset using SQLite. Each query addresses a real business reporting
need and is saved as a standalone `.sql` file in the `sql/` folder
for independent use by operations teams.

### Queries Covered
1. VLE Engagement Summary by Module
2. At-Risk Students by Assessment Threshold
3. Withdrawal Rates by Demographic Group
4. Student Activity Ranking by Module

---

## 1. Setup — Load Data into SQLite

In [1]:
import pandas as pd
import sqlite3
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Load cleaned data ──────────────────────────────────────────────
processed_path = '../data/processed/'

student_info       = pd.read_csv(processed_path + 'studentInfo_clean.csv')
student_vle        = pd.read_csv(processed_path + 'studentVle_sampled.csv')
student_assessment = pd.read_csv(processed_path + 'studentAssessment_clean.csv')
assessments        = pd.read_csv(processed_path + 'assessments_clean.csv')
student_reg        = pd.read_csv(processed_path + 'studentRegistration_clean.csv')
courses            = pd.read_csv(processed_path + 'courses_clean.csv')

# ── Create in-memory SQLite database ──────────────────────────────
conn = sqlite3.connect(':memory:')

student_info.to_sql('student_info', conn, index=False, if_exists='replace')
student_vle.to_sql('student_vle', conn, index=False, if_exists='replace')
student_assessment.to_sql('student_assessment', conn, index=False, if_exists='replace')
assessments.to_sql('assessments', conn, index=False, if_exists='replace')
student_reg.to_sql('student_reg', conn, index=False, if_exists='replace')
courses.to_sql('courses', conn, index=False, if_exists='replace')

print("SQLite database created successfully\n")
print("Tables loaded:")
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables.to_string(index=False))

SQLite database created successfully

Tables loaded:
              name
      student_info
       student_vle
student_assessment
       assessments
       student_reg
           courses


---

## 2. Helper Function

A helper function to run SQL queries, display results cleanly,
and save each query to the `sql/` folder as a standalone `.sql` file.

In [2]:
# Helper function — run query, display results, save to sql/ folder
sql_path = '../sql/'
os.makedirs(sql_path, exist_ok=True)

def run_query(filename, query, description):
    print(f"=== {description} ===\n")
    
    # Execute query
    result = pd.read_sql(query, conn)
    print(result.to_string(index=False))
    print(f"\nRows returned: {len(result):,}")
    
    # Save to sql/ folder
    filepath = sql_path + filename
    with open(filepath, 'w') as f:
        f.write(f"-- {description}\n")
        f.write(f"-- Project: Programme Health Intelligence\n")
        f.write(f"-- Author: Megha Sinha\n")
        f.write(f"-- Date: June 2026\n\n")
        f.write(query)
    print(f"\nQuery saved to sql/{filename}")
    
    return result

print("Helper function loaded successfully")

Helper function loaded successfully


---

## 3. Query 1 — VLE Engagement Summary by Module

**Business need:** Operations teams need a quick, repeatable summary
of engagement levels across modules — total clicks, average clicks
per student, and how that compares across cohorts.

**SQL concepts demonstrated:** GROUP BY, aggregation functions
(SUM, AVG, COUNT), ORDER BY, ROUND

In [3]:
query1 = """
SELECT
    si.code_module,
    si.code_presentation,
    COUNT(DISTINCT sv.id_student)          AS active_students,
    SUM(sv.sum_click)                      AS total_clicks,
    ROUND(AVG(sv.sum_click), 1)            AS avg_clicks_per_session,
    ROUND(SUM(sv.sum_click) * 1.0 /
          COUNT(DISTINCT sv.id_student), 1) AS avg_clicks_per_student
FROM student_vle sv
JOIN student_info si
    ON  sv.id_student        = si.id_student
    AND sv.code_module       = si.code_module
    AND sv.code_presentation = si.code_presentation
GROUP BY
    si.code_module,
    si.code_presentation
ORDER BY
    avg_clicks_per_student DESC
"""

result1 = run_query(
    'engagement_summary.sql',
    query1,
    'VLE Engagement Summary by Module'
)

=== VLE Engagement Summary by Module ===

code_module code_presentation  active_students  total_clicks  avg_clicks_per_session  avg_clicks_per_student
        FFF             2013B             1483        419950                    4.40                  283.20
        FFF             2014J             2039        525726                    4.30                  257.80
        FFF             2013J             2042        512601                    4.40                  251.00
        FFF             2014B             1317        294643                    4.30                  223.70
        AAA             2013J              376         64193                    3.50                  170.70
        AAA             2014J              353         59007                    3.50                  167.20
        EEE             2013J              922        152529                    4.30                  165.40
        EEE             2014J             1040        161863                    4.00  

---

## 4. Query 2 — At-Risk Students by Assessment Threshold

**Business need:** Operations teams need a list of students whose
median assessment score falls below the 55-mark threshold identified
in Notebook 4 — enabling targeted outreach to at-risk students.

**SQL concepts demonstrated:** subquery, GROUP BY, HAVING,
CASE WHEN, JOIN, filtering

In [4]:
query2 = """
SELECT
    si.id_student,
    si.code_module,
    si.code_presentation,
    si.final_result,
    ROUND(AVG(sa.score), 1)          AS median_score,
    COUNT(sa.id_assessment)          AS assessments_submitted,
    CASE
        WHEN AVG(sa.score) < 40 THEN 'Critical Risk'
        WHEN AVG(sa.score) < 55 THEN 'High Risk'
        WHEN AVG(sa.score) < 70 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END                              AS score_risk_flag
FROM student_info si
LEFT JOIN student_assessment sa
    ON  si.id_student        = sa.id_student
JOIN assessments a
    ON  sa.id_assessment     = a.id_assessment
    AND si.code_module       = a.code_module
WHERE si.final_result IN ('Fail', 'Withdrawn')
GROUP BY
    si.id_student,
    si.code_module,
    si.code_presentation,
    si.final_result
HAVING AVG(sa.score) < 55
ORDER BY
    median_score ASC
LIMIT 20
"""

result2 = run_query(
    'assessment_thresholds.sql',
    query2,
    'At-Risk Students by Assessment Threshold (sample of 20)'
)

=== At-Risk Students by Assessment Threshold (sample of 20) ===

 id_student code_module code_presentation final_result  median_score  assessments_submitted score_risk_flag
      53197         FFF             2013B         Fail          0.00                      1   Critical Risk
      96864         CCC             2014J    Withdrawn          0.00                      1   Critical Risk
     163962         FFF             2014B    Withdrawn          0.00                      1   Critical Risk
     242636         CCC             2014J         Fail          0.00                      1   Critical Risk
     244902         CCC             2014B    Withdrawn          0.00                      1   Critical Risk
     383311         CCC             2014B         Fail          0.00                      1   Critical Risk
     465764         CCC             2014B    Withdrawn          0.00                      1   Critical Risk
     465764         CCC             2014J    Withdrawn          0.00   

---

## 5. Query 3 — Withdrawal Rates by Demographic Group

**Business need:** Understanding which demographic groups have the
highest withdrawal rates helps target pre-enrolment support and
design more inclusive programmes.

**SQL concepts demonstrated:** LEFT JOIN across multiple tables,
ROUND, CAST, GROUP BY multiple columns, ORDER BY, calculated fields

In [5]:
query3 = """
SELECT
    si.age_band,
    si.gender,
    si.imd_band,
    COUNT(*)                                    AS total_students,
    SUM(CASE WHEN si.final_result = 'Withdrawn'
             THEN 1 ELSE 0 END)                 AS withdrawn_count,
    ROUND(
        SUM(CASE WHEN si.final_result = 'Withdrawn'
                 THEN 1 ELSE 0 END) * 100.0
        / COUNT(*), 1)                          AS withdrawal_rate_pct,
    ROUND(
        SUM(CASE WHEN si.final_result IN ('Pass', 'Distinction')
                 THEN 1 ELSE 0 END) * 100.0
        / COUNT(*), 1)                          AS success_rate_pct
FROM student_info si
GROUP BY
    si.age_band,
    si.gender,
    si.imd_band
HAVING COUNT(*) >= 50
ORDER BY
    withdrawal_rate_pct DESC
LIMIT 20
"""

result3 = run_query(
    'withdrawal_by_demographic.sql',
    query3,
    'Withdrawal Rates by Demographic Group (top 20)'
)

=== Withdrawal Rates by Demographic Group (top 20) ===

age_band gender imd_band  total_students  withdrawn_count  withdrawal_rate_pct  success_rate_pct
    0-35      M    0-10%            1213              503                41.50             32.60
    0-35      F   20-30%            1321              506                38.30             39.40
    0-35      M    10-20            1332              495                37.20             35.60
    0-35      M   20-30%            1372              509                37.10             37.60
   35-55      M    0-10%             392              141                36.00             37.20
   35-55      M    10-20             455              164                36.00             39.10
    0-35      F    0-10%            1296              456                35.20             35.60
    0-35      F    10-20            1286              450                35.00             39.60
    0-35      M   40-50%            1225              411              

---

## 6. Query 4 — Student Activity Ranking by Module

**Business need:** Operations teams need to rank students within
each module by engagement level — identifying the least engaged
students in each cohort for targeted outreach.

**SQL concepts demonstrated:** Window functions — RANK() OVER
(PARTITION BY ... ORDER BY ...), subquery, JOIN

---

## 7. Notebook Summary

### What This Notebook Demonstrated

This notebook loaded all cleaned OULAD data into an in-memory SQLite
database and executed four operational SQL queries — each saved as a
standalone `.sql` file in the `sql/` folder.

### SQL Concepts Demonstrated

| Query | File | Key SQL Concepts |
|---|---|---|
| VLE Engagement Summary | engagement_summary.sql | GROUP BY, SUM, AVG, COUNT, JOIN, ORDER BY |
| At-Risk by Assessment | assessment_thresholds.sql | LEFT JOIN, CASE WHEN, HAVING, subquery, AVG |
| Withdrawal by Demographic | withdrawal_by_demographic.sql | Multiple GROUP BY, calculated fields, ROUND, CAST |
| Student Activity Ranking | student_activity_ranking.sql | Window functions — RANK() OVER PARTITION BY, subquery |

### Key Observations from Query Results

- **FFF** consistently produces the highest engagement per student
  (223–283 avg clicks) across all four presentations
- **CCC** dominates the at-risk assessment list — 14 of 20 lowest
  scoring students are from CCC modules
- **Demographic patterns are observable but correlational** — students
aged 0-35, male, in the 0-10% IMD band show the highest withdrawal
rate (41.5%). This association warrants further investigation but
cannot be attributed to causation without additional qualitative
and contextual data.
- **Window functions** allow operations teams to rank students within
  each module cohort — enabling targeted outreach to the least engaged
  students in each group

### Files Produced

| File | Location |
|---|---|
| engagement_summary.sql | sql/ |
| assessment_thresholds.sql | sql/ |
| withdrawal_by_demographic.sql | sql/ |
| student_activity_ranking.sql | sql/ |